# Pan-cancer MI-4: fibroblast-to-tumor LR perturbation

This notebook evaluates fixed-model, in silico ligand-receptor (LR) perturbations associated with fibroblast-to-tumor meta-interaction 4 (MI-4).

## Workflow

1. Load the processed atlas, reference LR loadings, edge-level MI activities, and a trained 11-MI SpiderNet checkpoint.
2. Select the five highest-loading usable MI-4 LR pairs and one LR-pair-count-matched random control set (seed 123). The default control pool restricts MI-4 loading and gene overlap; diagnostics record any fallback.
3. Select tumor receivers of fibroblast-to-tumor edges with positive MI-4 activity. Reconstruct their unperturbed expression to define gene-wise z-score references.
4. With model weights fixed, set selected sender ligand and receiver receptor expression to zero or 50%, then calculate post-minus-pre invasion, angiogenesis, and hypoxia module scores.
5. Export cell-level changes, summaries, a cancer-type-faceted violin/box figure, and per-panel tests. The final analysis cell pairs top-LR and random-LR effects by receiver cell; an earlier summary also exports unpaired comparisons.

## Inputs and execution

Use a fresh Python kernel and run cells in order. Install SpiderNet and its PyTorch/PyG dependencies using the package README, together with the tutorial/Jupyter dependencies. Set the data and result paths in the configuration cell to the corresponding local files.

- **Processed atlas:** `ProcessedData_entire` containing `adata_all.h5ad`, `adata_list.pkl`, `SpiderNet_data_pyg_list.pkl` or `.pt`, `LR_list.pkl`, and `genenames_train.pkl`. AnnData gene order, graph expression columns, cell indices, and batch order must agree with the reference model and inference outputs.
- **Reference inference:** `Factor_envir_use.npy` (edges by MIs) and preferably `loading_LR_use.csv` from full-data inference in `Pancancer_analysis_V2.ipynb`, with a matching trained checkpoint. LR and gene metadata are resolved from the configured search directories.
- **CancerSEA markers:** tab-separated `Invasion.txt`, `Angiogenesis.txt`, and `Hypoxia.txt`, each with a `GeneName` column.

## Outputs and interpretation

Outputs are written under `output/lrko/` (or the configured output root): selected LR/gene/cell tables, control-pool and runtime diagnostics, long-form module-score changes, statistical summaries, and PDF/PNG figures. The default score averages gene z-scores relative to unperturbed reconstructed expression across all selected tumor receivers; within-cancer-type references are also supported. Negative changes indicate lower predicted program scores.

Perturbations modify expression in selected cells while retaining the full graph; the readout uses total reconstructed expression. CancerSEA programs are stored in the existing `GO_program` column. Rerunning analysis cells writes to the configured output filenames.


In [ ]:
# 1. Configuration and dependencies

from pathlib import Path
import os

# Shared data and model outputs remain upstream of this tutorial.
_root_candidates = [Path.cwd(), *Path.cwd().parents]
SPIDERNET_ROOT = Path(os.environ.get("SPIDERNET_ROOT", next(
    (str(p) for p in _root_candidates if (p / "Data").is_dir() and (p / "Results").is_dir()),
    str(Path.cwd()),
)))
pancancer_output_dir = Path(os.environ.get("SPIDERNET_PANCANCER_OUTPUT", "output"))
pancancer_output_dir.mkdir(parents=True, exist_ok=True)

import json
import gc
import math
import re
import time
import copy
from collections import OrderedDict

import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
from scipy.stats import mannwhitneyu, wilcoxon

# -----------------------------
# Core paths
# -----------------------------
DATA_ROOT = Path(os.environ.get("SPIDERNET_PANCANCER_DATA", SPIDERNET_ROOT / "Data/Pancancer"))
OUTPUT_ROOT = SPIDERNET_ROOT / "Results/Pancancer"

# Full pan-cancer processed bundle used for inference.
PROCESSED_DATA_DIR = Path(os.environ.get("SPIDERNET_PANCANCER_PROCESSED", OUTPUT_ROOT / "ProcessedData_entire"))
processed_data_dir = PROCESSED_DATA_DIR

# Reference model/result directory from Pancancer_analysis_V2 / model-training output.
# Must contain Factor_envir_use.npy and preferably loading_LR_use.csv.
REFERENCE_RESULTS_DIR = Path(os.environ.get("SPIDERNET_PANCANCER_RESULTS", OUTPUT_ROOT / "V1/SpiderNet_Result_dim11"))
LRKO_RUN_DIR = REFERENCE_RESULTS_DIR

# Reference processed-data directory that stores LR_list.pkl / genenames_train.pkl.
REFERENCE_PROCESSED_DATA_DIR = OUTPUT_ROOT / "ProcessedData"

# With None, select the highest parsed epoch; modification time breaks ties.
# An explicit checkpoint path fixes the model independently of directory contents.
TRAINED_MODEL_PATH = None

# AnnData fields used in the pan-cancer processed objects.
SAMPLE_COL = "SampleID"
CELL_CLASS_COL = "celltype_final"

# SpiderNet model dimension from the trained pan-cancer run.
DIM_ENVIR = 11

# Model and inference directories used by the loading helpers.
run_dirs = {
    "run_dir": LRKO_RUN_DIR,
    "model_dir": LRKO_RUN_DIR / "Model",
}

# -----------------------------
# LR-KO settings
# -----------------------------
PC_LRKO_MI = "MI4"
PC_LRKO_MI_INDEX = int(re.findall(r"\d+", PC_LRKO_MI)[-1]) - 1

# Select by descending raw LR loading among pairs with genes present on both sides.
PC_LRKO_TOP_N_LR = 5
PC_LRKO_RANDOM_SEED = 123
# Apply a strict greater-than threshold to stored edge-level MI-4 activity.
PC_LRKO_MI_STRENGTH_THRESHOLD = 0.0

# Sample one control set, matched by LR-pair count. The default pool uses the lower
# half of MI-4 loadings and excludes genes shared with top LR pairs or CancerSEA
# readouts. Section 4 records progressively relaxed criteria if the pool is too small.
PC_LRKO_RANDOM_BASELINE_MODE = "strict_low_MI4_no_overlap"
PC_LRKO_RANDOM_LOW_LOADING_QUANTILE = 0.50
PC_LRKO_RANDOM_EXCLUDE_TOP_LR_GENES = True
PC_LRKO_RANDOM_EXCLUDE_CANCERSEA_READOUT_GENES = True

PC_LRKO_SENDER_CELLTYPE = "Fibroblast"
PC_LRKO_RECEIVER_TUMOR_SUFFIX = "-cancercell"

PC_LRKO_CANCERSEA_PROGRAMS = ["Invasion", "Angiogenesis", "Hypoxia"]
PC_LRKO_CANCERSEA_PATH = DATA_ROOT / "CancerSEA_marker"

# References use unperturbed reconstructed expression:
#   "zscore_mean_global": gene-wise mean/std from all selected tumor receivers.
#   "zscore_mean_within_cancertype": gene-wise mean/std within each CancerType.
PC_LRKO_MODULE_SCORE_METHOD = "zscore_mean_global"

PC_LRKO_VALID_MODULE_SCORE_METHODS = {
    "zscore_mean_global",
    "zscore_mean_within_cancertype",
}
if PC_LRKO_MODULE_SCORE_METHOD not in PC_LRKO_VALID_MODULE_SCORE_METHODS:
    raise ValueError(
        f"Unsupported PC_LRKO_MODULE_SCORE_METHOD={PC_LRKO_MODULE_SCORE_METHOD}. "
        f"Choose from {sorted(PC_LRKO_VALID_MODULE_SCORE_METHODS)}."
    )

# Set to True only if you need the selected-edge table; it can be very large.
PC_LRKO_SAVE_SELECTED_EDGE_TABLE = False

# Output directory for this standalone LR-KO analysis.
pc_mi4_lrko_outdir = pancancer_output_dir / "lrko"
pc_mi4_lrko_outdir.mkdir(parents=True, exist_ok=True)

# Device.
device = "cuda" if torch.cuda.is_available() else "cpu"
model = None

print("Using device:", device)
print("Processed data directory:", processed_data_dir)
print("Reference/result directory:", LRKO_RUN_DIR)
print("Output directory:", pc_mi4_lrko_outdir)
print("MI:", PC_LRKO_MI, "| MI index:", PC_LRKO_MI_INDEX)
print("CancerSEA programs:", PC_LRKO_CANCERSEA_PROGRAMS)
print("Module-score method:", PC_LRKO_MODULE_SCORE_METHOD)


In [ ]:
# 2. Resolve reference inputs and validate the processed atlas

from SpiderNet.io import load_processed_data, spidernet_pyg_list_exists


def _pc_resolve_existing_reference_file(search_dirs, candidates, label, required=True):
    if isinstance(candidates, (str, Path)):
        candidates = [candidates]

    checked = []
    for directory in search_dirs:
        directory = Path(directory)
        for candidate in candidates:
            candidate_path = directory / candidate
            checked.append(candidate_path)
            if candidate_path.exists():
                print(f"Resolved {label}: {candidate_path}")
                return candidate_path

    if required:
        checked_text = "\n  - " + "\n  - ".join(str(p) for p in checked)
        raise FileNotFoundError(f"Cannot find {label}. Checked:{checked_text}")
    return None


def _pc_resolve_model_checkpoint(reference_results_dir, explicit_model_path=None):
    reference_results_dir = Path(reference_results_dir)

    if explicit_model_path is not None:
        explicit_model_path = Path(explicit_model_path)
        if not explicit_model_path.exists():
            raise FileNotFoundError(f"Model checkpoint does not exist: {explicit_model_path}")
        return explicit_model_path

    search_dirs = []
    if (reference_results_dir / "Model").exists():
        search_dirs.append(reference_results_dir / "Model")
    search_dirs.append(reference_results_dir)

    candidates = []
    for search_dir in search_dirs:
        for pattern in ["model_epoch*", "*checkpoint*", "*.pt", "*.pth", "*.pkl"]:
            candidates.extend(search_dir.glob(pattern))

    candidates = [p for p in candidates if p.is_file()]
    if len(candidates) == 0:
        raise FileNotFoundError(
            "Cannot find a trained model checkpoint under "
            f"{reference_results_dir} or {reference_results_dir / 'Model'}"
        )

    def _extract_epoch(path):
        match = re.search(r"epoch(\d+)", path.stem)
        return int(match.group(1)) if match else -1

    candidates = sorted(candidates, key=lambda p: (_extract_epoch(p), p.stat().st_mtime))
    return candidates[-1]


# The first existing metadata file wins; its provenance must match the checkpoint.
REFERENCE_FEATURE_SEARCH_DIRS = [
    REFERENCE_RESULTS_DIR,
    REFERENCE_PROCESSED_DATA_DIR,
    PROCESSED_DATA_DIR,
]

reference_lr_list_path = _pc_resolve_existing_reference_file(
    REFERENCE_FEATURE_SEARCH_DIRS,
    "LR_list.pkl",
    label="reference LR list",
    required=True,
)

reference_genename_path = _pc_resolve_existing_reference_file(
    REFERENCE_FEATURE_SEARCH_DIRS,
    ["genenames_train.pkl", "genenames.pkl"],
    label="reference gene-name file",
    required=True,
)

reference_model_path = _pc_resolve_model_checkpoint(
    REFERENCE_RESULTS_DIR,
    TRAINED_MODEL_PATH,
)
print("Resolved model checkpoint:", reference_model_path)

required_processed_files = [
    "adata_all.h5ad",
    "adata_list.pkl",
    "LR_list.pkl",
    "genenames_train.pkl",
]
missing_processed_files = [
    f for f in required_processed_files
    if not (processed_data_dir / f).exists()
]
if missing_processed_files or not spidernet_pyg_list_exists(processed_data_dir):
    raise FileNotFoundError(
        f"ProcessedData is incomplete. Missing files under {processed_data_dir}: "
        f"{missing_processed_files}. Run the pan-cancer preprocessing/inference notebook first."
    )

processed = load_processed_data(processed_data_dir)

reference_lr_list = pd.read_pickle(reference_lr_list_path)
reference_genenames = pd.read_pickle(reference_genename_path)
reference_genenames_array = np.asarray(reference_genenames).astype(str)
processed_genenames_array = np.asarray(processed.genenames_train).astype(str)

print("Number of batches:", len(processed.spidernet_data))
print("Number of LR pairs in full processed data:", len(processed.lr_list))
print("Number of LR pairs in reference training run:", len(reference_lr_list))
print("Number of training genes in full processed data:", processed.genenames_train.shape[0])
print("Number of training genes in reference training run:", reference_genenames_array.shape[0])

if processed.genenames_train.shape[0] != reference_genenames_array.shape[0]:
    raise ValueError(
        "The full processed data and the reference training model use different numbers "
        f"of genes: full={processed.genenames_train.shape[0]}, "
        f"reference={reference_genenames_array.shape[0]}. "
        "Rerun full-data preprocessing with the reference genenames_train.pkl."
    )

if not np.array_equal(processed_genenames_array, reference_genenames_array):
    raise ValueError(
        "The full processed data gene order does not match the reference training gene order. "
        "Rerun full-data preprocessing with the reference genenames_train.pkl."
    )

if len(processed.lr_list) != len(reference_lr_list):
    print(
        "WARNING: full processed LR list length differs from the reference training LR list "
        f"({len(processed.lr_list)} vs {len(reference_lr_list)}). "
        "LR loading interpretation will use reference LR metadata / exported loading file."
    )

# Check cell counts against the largest graph index in each batch.
summary_rows = []
for i in range(len(processed.adata_list)):
    adata_i = processed.adata_list[i]
    edge_index_i = processed.spidernet_data[i]["edge_index"]
    edge_index_max = int(torch.max(edge_index_i).cpu().item()) if hasattr(edge_index_i, "cpu") else int(np.max(edge_index_i))
    summary_rows.append({
        "batch_index": i,
        "n_cells": int(adata_i.n_obs),
        "n_genes": int(adata_i.n_vars),
        "edge_index_matches_n_cells": bool(adata_i.n_obs == edge_index_max + 1),
    })
summary_df = pd.DataFrame(summary_rows)
display(summary_df.head())

if not summary_df["edge_index_matches_n_cells"].all():
    raise ValueError("At least one batch has an edge-index / cell-count mismatch.")

print("Processed data sanity check passed.")

del reference_genenames, reference_genenames_array, processed_genenames_array, summary_rows, summary_df
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()


In [ ]:
# 3. Utilities for graph metadata, LR genes, and score references
def _pc_lrko_to_numpy(x):
    if hasattr(x, "detach"):
        x = x.detach()
    if hasattr(x, "cpu"):
        x = x.cpu()
    if sp.issparse(x):
        return x.toarray()
    return np.asarray(x)


def _pc_lrko_get_field(obj, key):
    if isinstance(obj, dict):
        return obj[key]
    if hasattr(obj, key):
        return getattr(obj, key)
    return obj[key]


def _pc_lrko_edge_index_to_numpy(edge_index_obj):
    edge_index = _pc_lrko_to_numpy(edge_index_obj).astype(np.int64, copy=False)

    if edge_index.ndim != 2:
        raise ValueError(f"edge_index should be 2D, got shape {edge_index.shape}")

    if edge_index.shape[1] == 2:
        return edge_index

    if edge_index.shape[0] == 2:
        return edge_index.T.astype(np.int64, copy=False)

    raise ValueError(f"edge_index should have shape [E, 2] or [2, E], got {edge_index.shape}")


def _pc_lrko_edge_count_from_data(data_obj):
    edge_index = _pc_lrko_edge_index_to_numpy(_pc_lrko_get_field(data_obj, "edge_index"))
    return int(edge_index.shape[0])


def _pc_lrko_get_celltypes(adata, data_obj):
    preferred_cols = [
        globals().get("CELL_CLASS_COL", None),
        "celltype_final",
        "cell_type_final",
        "celltype",
        "cell_type",
        "cell_class",
    ]

    for col in preferred_cols:
        if col is not None and col in adata.obs.columns:
            return adata.obs[col].astype(str).to_numpy()

    if isinstance(data_obj, dict) and "cell_class" in data_obj:
        return _pc_lrko_to_numpy(data_obj["cell_class"]).astype(str)

    if hasattr(data_obj, "cell_class"):
        return _pc_lrko_to_numpy(data_obj.cell_class).astype(str)

    raise KeyError(
        "Cannot find cell-type labels. Expected adata.obs[CELL_CLASS_COL], "
        "adata.obs['celltype_final'], or data_obj['cell_class']."
    )


def _pc_lrko_get_sample_id(adata, sample_index):
    sample_col_candidates = [
        globals().get("SAMPLE_COL", None),
        "SampleID",
        "sample_id",
        "sample",
        "samples",
        "Sample",
        "slice",
        "slide",
        "library_id",
    ]

    # Use the first observed label; one sample label per batch is expected.
    for col in sample_col_candidates:
        if col is not None and col in adata.obs.columns:
            vals = pd.Series(adata.obs[col]).dropna().astype(str).unique()
            if len(vals) > 0:
                return str(vals[0])

    return f"sample_{sample_index}"


def _pc_lrko_extract_cancer_type_from_tumor_celltype(celltype):
    s = str(celltype)
    if PC_LRKO_RECEIVER_TUMOR_SUFFIX not in s:
        return None
    return s.replace(PC_LRKO_RECEIVER_TUMOR_SUFFIX, "").strip(" -_")


def _pc_lrko_get_barcodes(adata, cell_indices):
    barcode_candidates = ["barcode", "Barcode", "cell_id", "cell", "CellID"]

    for col in barcode_candidates:
        if col in adata.obs.columns:
            return adata.obs.iloc[cell_indices][col].astype(str).to_numpy()

    return adata.obs_names[cell_indices].astype(str).to_numpy()


def _pc_lrko_resolve_gene_indices(var_names, genes):
    var_names = pd.Index([str(x) for x in var_names])
    upper_to_idx = {}

    for i, g in enumerate(var_names):
        upper_to_idx.setdefault(str(g).upper(), i)

    idx = []
    used = []
    missing = []

    for g in genes:
        g = str(g).strip()
        if len(g) == 0:
            continue

        idx_cur = None
        if g in var_names:
            loc = var_names.get_loc(g)
            if isinstance(loc, slice):
                idx_cur = int(loc.start)
            elif isinstance(loc, np.ndarray):
                idx_cur = int(np.where(loc)[0][0])
            else:
                idx_cur = int(loc)
        elif g.upper() in upper_to_idx:
            idx_cur = int(upper_to_idx[g.upper()])

        if idx_cur is None:
            missing.append(g)
        else:
            idx.append(idx_cur)
            used.append(str(var_names[idx_cur]))

    seen = set()
    idx_unique = []
    used_unique = []

    for i, g in zip(idx, used):
        if i not in seen:
            seen.add(i)
            idx_unique.append(i)
            used_unique.append(g)

    return np.asarray(idx_unique, dtype=np.int64), used_unique, missing


def _pc_lrko_load_cancersea_gene_sets(programs, cancersea_path, var_names):
    gene_sets = OrderedDict()

    for program in programs:
        path = Path(cancersea_path) / f"{program}.txt"
        if not path.exists():
            raise FileNotFoundError(f"Cannot find CancerSEA marker file: {path}")

        df = pd.read_csv(path, sep="\t", header=0)

        if "GeneName" not in df.columns:
            raise KeyError(f"{path} does not contain a 'GeneName' column.")

        genes_raw = (
            df["GeneName"]
            .dropna()
            .astype(str)
            .str.strip()
            .loc[lambda x: x.ne("")]
            .drop_duplicates()
            .tolist()
        )

        gene_idx, genes_used, genes_missing = _pc_lrko_resolve_gene_indices(var_names, genes_raw)

        if len(gene_idx) == 0:
            raise ValueError(f"No genes from CancerSEA {program} are found in adata.var_names.")

        gene_sets[program] = {
            "genes_raw": genes_raw,
            "gene_idx": gene_idx,
            "genes_used": genes_used,
            "genes_missing": genes_missing,
        }

    return gene_sets


def _pc_lrko_build_gene_reference(gene_sum, gene_sumsq, gene_n):
    if int(gene_n) <= 0:
        raise ValueError("Cannot build gene reference with gene_n <= 0.")

    # Population moments of the unperturbed receiver-cell reconstructions.
    mean = gene_sum / float(gene_n)
    var = (gene_sumsq / float(gene_n)) - mean ** 2
    std = np.sqrt(np.maximum(var, 1e-8))

    std[~np.isfinite(std)] = 1.0
    std[std < 1e-6] = 1.0

    return {
        "mean": mean.astype(np.float32),
        "std": std.astype(np.float32),
        "n_ref": int(gene_n),
    }


def _pc_lrko_make_cell_key(sample_index, barcode):
    return f"sample{int(sample_index)}::{barcode}"


print("Output directory:", pc_mi4_lrko_outdir)
print("MI:", PC_LRKO_MI, "| MI index:", PC_LRKO_MI_INDEX)
print("CancerSEA programs:", PC_LRKO_CANCERSEA_PROGRAMS)
print("Module-score method:", PC_LRKO_MODULE_SCORE_METHOD)


In [ ]:
# 4. Select top MI-4 LR pairs and the random control

def _pc_lrko_mi_name_candidates(mi_name):
    mi_name = str(mi_name)
    digits = re.findall(r"\d+", mi_name)

    candidates = [
        mi_name,
        mi_name.replace("_", "-"),
        mi_name.replace("-", "_"),
    ]

    if len(digits) > 0:
        d = digits[-1]
        candidates.extend([
            f"MI{d}",
            f"MI-{d}",
            f"MI_{d}",
            f"Meta-interaction {d}",
            f"meta-interaction {d}",
            str(d),
        ])

    out = []
    for x in candidates:
        if x not in out:
            out.append(x)

    return out


def _pc_lrko_extract_mi_lr_vector(df, mi_name):
    candidates = _pc_lrko_mi_name_candidates(mi_name)

    # Preferred orientation: rows = MIs, columns = LR pairs.
    for cand in candidates:
        if cand in df.index:
            return df.loc[cand, :].copy(), "rows_are_MIs", cand

    # Fallback orientation: rows = LR pairs, columns = MIs.
    for cand in candidates:
        if cand in df.columns:
            return df.loc[:, cand].copy(), "columns_are_MIs", cand

    raise ValueError(
        f"Cannot find {mi_name!r} in LR loading matrix.\n"
        f"Tried candidates: {candidates}\n"
        f"First 10 rows: {list(df.index[:10])}\n"
        f"First 10 columns: {list(df.columns[:10])}"
    )


def _pc_lrko_lr_record_to_name(record):
    if isinstance(record, str):
        return record

    if isinstance(record, (tuple, list, np.ndarray)) and len(record) >= 2:
        lig = record[0]
        rec = record[1]
        if isinstance(lig, (tuple, list, np.ndarray, set)):
            lig = "+".join([str(x) for x in lig])
        if isinstance(rec, (tuple, list, np.ndarray, set)):
            rec = "+".join([str(x) for x in rec])
        return f"{lig} -> {rec}"

    if isinstance(record, dict):
        ligand_keys = ["ligand", "Ligand", "source", "Source", "ligand_gene_symbol", "ligand_symbol"]
        receptor_keys = ["receptor", "Receptor", "target", "Target", "receptor_gene_symbol", "receptor_symbol"]
        ligand_key = next((k for k in ligand_keys if k in record), None)
        receptor_key = next((k for k in receptor_keys if k in record), None)
        if ligand_key is not None and receptor_key is not None:
            return f"{record[ligand_key]} -> {record[receptor_key]}"

    raise ValueError(f"Unsupported LR-list record format: {type(record)} | {record}")


def _pc_lrko_get_lr_names_from_reference():
    if "reference_lr_list_path" in globals() and reference_lr_list_path is not None and Path(reference_lr_list_path).exists():
        lr_obj = pd.read_pickle(reference_lr_list_path)
    elif "processed" in globals() and hasattr(processed, "lr_list"):
        lr_obj = processed.lr_list
    else:
        raise FileNotFoundError(
            "Cannot infer LR names because neither reference_lr_list_path nor processed.lr_list is available."
        )

    if isinstance(lr_obj, pd.DataFrame):
        arrow_cols = [
            c for c in lr_obj.columns
            if lr_obj[c].astype(str).str.contains(r"\s*->\s*", regex=True).any()
        ]
        if len(arrow_cols) > 0:
            return lr_obj[arrow_cols[0]].astype(str).tolist()

        ligand_candidates = [
            "ligand", "Ligand", "source", "Source",
            "ligand_gene_symbol", "ligand_symbol", "interaction_name"
        ]
        receptor_candidates = [
            "receptor", "Receptor", "target", "Target",
            "receptor_gene_symbol", "receptor_symbol", "partner"
        ]
        ligand_col = next((c for c in ligand_candidates if c in lr_obj.columns), None)
        receptor_col = next((c for c in receptor_candidates if c in lr_obj.columns), None)
        if ligand_col is None or receptor_col is None:
            raise ValueError(
                "Could not infer ligand/receptor columns from LR DataFrame. "
                f"Available columns: {list(lr_obj.columns)}"
            )
        return [
            f"{lig} -> {rec}"
            for lig, rec in zip(lr_obj[ligand_col].astype(str), lr_obj[receptor_col].astype(str))
        ]

    return [_pc_lrko_lr_record_to_name(x) for x in list(lr_obj)]


def _pc_lrko_load_lr_loading_matrix(run_dir):
    run_dir = Path(run_dir)

    # Interactive loading matrices take precedence over files when present.
    if "loading_LR_use" in globals():
        df = loading_LR_use.copy()
        source = "existing variable loading_LR_use"
    elif "Loading_LR_use" in globals():
        df = Loading_LR_use.copy()
        source = "existing variable Loading_LR_use"
    else:
        candidate_csv = [
            run_dir / "loading_LR_use.csv",
            run_dir / "Loading_LR_use.csv",
            run_dir / "LR_loading_use.csv",
            run_dir / "loading_LR.csv",
            run_dir / "LR_loading.csv",
        ]

        csv_path = next((p for p in candidate_csv if p.exists()), None)
        if csv_path is not None:
            df = pd.read_csv(csv_path, index_col=0)
            source = str(csv_path)
        else:
            candidate_npy = [
                run_dir / "loading_LR_use.npy",
                run_dir / "Loading_LR_use.npy",
                run_dir / "LR_loading_use.npy",
            ]
            npy_path = next((p for p in candidate_npy if p.exists()), None)
            if npy_path is None:
                raise FileNotFoundError(
                    "Cannot find LR-pair loading file. Tried:\n"
                    + "\n".join([str(p) for p in candidate_csv + candidate_npy])
                )

            arr = np.load(npy_path)
            lr_names = _pc_lrko_get_lr_names_from_reference()

            if arr.shape[1] == len(lr_names):
                mi_names = [f"MI{i+1}" for i in range(arr.shape[0])]
                df = pd.DataFrame(arr, index=mi_names, columns=lr_names)
            elif arr.shape[0] == len(lr_names):
                mi_names = [f"MI{i+1}" for i in range(arr.shape[1])]
                df = pd.DataFrame(arr, index=lr_names, columns=mi_names)
            else:
                df = pd.DataFrame(arr)

            source = str(npy_path)

    df = pd.DataFrame(df)
    df = df.apply(pd.to_numeric, errors="coerce")
    return df, source


def _pc_lrko_parse_lr_pair_name(lr_name):
    lr_name = str(lr_name)

    if re.search(r"\s*->\s*", lr_name):
        parts = re.split(r"\s*->\s*", lr_name)
    elif re.search(r"\s*[-_]\s*", lr_name) and lr_name.count(" ") == 0:
        # Compact-name fallback, splitting at the first hyphen or underscore.
        parts = re.split(r"[-_]", lr_name, maxsplit=1)
    else:
        raise ValueError(
            f"Cannot parse LR pair name {lr_name!r}. Expected format like 'LIG -> REC' or 'LIG1+LIG2 -> REC1+REC2'."
        )

    if len(parts) != 2:
        raise ValueError(f"Cannot parse LR pair name {lr_name!r} into exactly two sides.")

    ligands = [x.strip() for x in re.split(r"\+", parts[0]) if x.strip() != ""]
    receptors = [x.strip() for x in re.split(r"\+", parts[1]) if x.strip() != ""]

    if len(ligands) == 0 or len(receptors) == 0:
        raise ValueError(f"LR pair {lr_name!r} has empty ligand or receptor side.")

    return list(dict.fromkeys(ligands)), list(dict.fromkeys(receptors))


def _pc_lrko_lr_genes_to_indices(lr_pair_names, var_names):
    ligand_genes = []
    receptor_genes = []
    parse_errors = []

    for lr_name in lr_pair_names:
        try:
            ligs, recs = _pc_lrko_parse_lr_pair_name(lr_name)
            ligand_genes.extend(ligs)
            receptor_genes.extend(recs)
        except Exception as e:
            parse_errors.append({"LR_pair": str(lr_name), "error": repr(e)})

    ligand_genes = sorted(set(ligand_genes))
    receptor_genes = sorted(set(receptor_genes))

    ligand_idx, ligand_found, ligand_missing = _pc_lrko_resolve_gene_indices(var_names, ligand_genes)
    receptor_idx, receptor_found, receptor_missing = _pc_lrko_resolve_gene_indices(var_names, receptor_genes)

    return {
        "ligand_genes": ligand_genes,
        "receptor_genes": receptor_genes,
        "ligand_idx": ligand_idx,
        "receptor_idx": receptor_idx,
        "ligand_found": ligand_found,
        "receptor_found": receptor_found,
        "ligand_missing": ligand_missing,
        "receptor_missing": receptor_missing,
        "parse_errors": parse_errors,
    }


def _pc_lrko_is_usable_lr_pair(lr_name, var_names):
    try:
        genes = _pc_lrko_lr_genes_to_indices([lr_name], var_names)
        return len(genes["ligand_idx"]) > 0 and len(genes["receptor_idx"]) > 0
    except Exception:
        return False


# -----------------------------
# Load LR loading vector for MI4
# -----------------------------
pc_lrko_loading_LR_df, pc_lrko_loading_source = _pc_lrko_load_lr_loading_matrix(run_dirs["run_dir"])
pc_lrko_lr_vec, pc_lrko_orientation, pc_lrko_mi_name_used = _pc_lrko_extract_mi_lr_vector(
    pc_lrko_loading_LR_df,
    PC_LRKO_MI,
)

pc_lrko_lr_vec = (
    pd.to_numeric(pc_lrko_lr_vec, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)

metadata_like_names = {"mi", "mi_id", "pathway", "pathway_name", "annotation", "name", "index"}
pc_lrko_lr_vec = pc_lrko_lr_vec[
    ~pc_lrko_lr_vec.index.astype(str).str.lower().isin(metadata_like_names)
]

if pc_lrko_lr_vec.shape[0] == 0:
    raise ValueError(f"No finite LR-pair loading values found for {PC_LRKO_MI}.")

pc_lrko_gene_names = pd.Index([str(x) for x in processed.adata_list[0].var_names])

# Take up to PC_LRKO_TOP_N_LR highest-loading usable pairs. A usable pair has
# at least one ligand and receptor gene in the panel; all subunits need not be present.
pc_lrko_sorted_lr = pc_lrko_lr_vec.sort_values(ascending=False)
pc_lrko_top_pair_names = []
for lr_name in pc_lrko_sorted_lr.index.astype(str):
    if _pc_lrko_is_usable_lr_pair(lr_name, pc_lrko_gene_names):
        pc_lrko_top_pair_names.append(str(lr_name))
    if len(pc_lrko_top_pair_names) >= PC_LRKO_TOP_N_LR:
        break

if len(pc_lrko_top_pair_names) == 0:
    raise ValueError(
        f"No top {PC_LRKO_MI} LR pair could be parsed and resolved in adata.var_names."
    )

pc_lrko_top_gene_info = _pc_lrko_lr_genes_to_indices(pc_lrko_top_pair_names, pc_lrko_gene_names)

if len(pc_lrko_top_gene_info["ligand_idx"]) == 0 or len(pc_lrko_top_gene_info["receptor_idx"]) == 0:
    raise ValueError(
        "Top MI4 LR genes could not be resolved in adata.var_names. "
        f"Missing ligands: {pc_lrko_top_gene_info['ligand_missing']}; "
        f"missing receptors: {pc_lrko_top_gene_info['receptor_missing']}"
    )

# Sample one LR-pair-count-matched control set using a dedicated NumPy generator.
# Fallbacks relax the loading cutoff, then readout-gene exclusion, then top-gene
# exclusion. Exported diagnostics identify the criteria actually used.
rng = np.random.default_rng(PC_LRKO_RANDOM_SEED)

pc_lrko_top_gene_idx_set = set(map(int, pc_lrko_top_gene_info["ligand_idx"])) | set(map(int, pc_lrko_top_gene_info["receptor_idx"]))

# Resolve CancerSEA genes for control-pool exclusion; the first pass reloads
# these gene sets to define reconstruction readouts.
pc_lrko_readout_gene_idx_set_for_random = set()
pc_lrko_readout_genes_for_random = []
if bool(globals().get("PC_LRKO_RANDOM_EXCLUDE_CANCERSEA_READOUT_GENES", True)):
    try:
        pc_lrko_cancersea_gene_sets_for_random = _pc_lrko_load_cancersea_gene_sets(
            programs=PC_LRKO_CANCERSEA_PROGRAMS,
            cancersea_path=PC_LRKO_CANCERSEA_PATH,
            var_names=processed.adata_list[0].var_names,
        )
        for _program, _info in pc_lrko_cancersea_gene_sets_for_random.items():
            pc_lrko_readout_gene_idx_set_for_random.update(map(int, _info["gene_idx"]))
            pc_lrko_readout_genes_for_random.extend(_info["genes_used"])
        pc_lrko_readout_genes_for_random = sorted(set(pc_lrko_readout_genes_for_random))
    except Exception as e:
        print("[Warning] Could not load CancerSEA genes for random-baseline exclusion:", repr(e))
        pc_lrko_readout_gene_idx_set_for_random = set()
        pc_lrko_readout_genes_for_random = []


def _pc_lrko_lr_pair_diagnostics(lr_name):
    info = _pc_lrko_lr_genes_to_indices([lr_name], pc_lrko_gene_names)
    gene_idx = set(map(int, info["ligand_idx"])) | set(map(int, info["receptor_idx"]))
    return {
        "usable": len(info["ligand_idx"]) > 0 and len(info["receptor_idx"]) > 0,
        "gene_idx": gene_idx,
        "overlap_top_gene": len(gene_idx & pc_lrko_top_gene_idx_set) > 0,
        "overlap_readout_gene": len(gene_idx & pc_lrko_readout_gene_idx_set_for_random) > 0,
        "ligand_genes_found": info["ligand_found"],
        "receptor_genes_found": info["receptor_found"],
    }


def _pc_lrko_build_random_candidate_pool(require_low_loading=True, exclude_top_genes=True, exclude_readout_genes=True):
    pool = []
    loading_threshold = None

    if require_low_loading:
        q = float(globals().get("PC_LRKO_RANDOM_LOW_LOADING_QUANTILE", 0.50))
        q = min(max(q, 0.0), 1.0)
        loading_threshold = float(np.nanquantile(pc_lrko_lr_vec.values.astype(float), q))

    top_pair_set = set(pc_lrko_top_pair_names)

    for lr_name in pc_lrko_lr_vec.index.astype(str):
        lr_name = str(lr_name)
        if lr_name in top_pair_set:
            continue

        raw_loading = float(pc_lrko_lr_vec.loc[lr_name])
        if require_low_loading and raw_loading > loading_threshold:
            continue

        try:
            diag = _pc_lrko_lr_pair_diagnostics(lr_name)
        except Exception:
            continue

        if not diag["usable"]:
            continue
        if exclude_top_genes and diag["overlap_top_gene"]:
            continue
        if exclude_readout_genes and diag["overlap_readout_gene"]:
            continue

        pool.append(lr_name)

    return pool, loading_threshold


def _pc_lrko_sample_random_pairs_strict():
    n_need = len(pc_lrko_top_pair_names)
    mode = str(globals().get("PC_LRKO_RANDOM_BASELINE_MODE", "strict_low_MI4_no_overlap"))

    attempts = []

    if mode == "legacy":
        attempts.append({
            "label": "legacy_non_top_usable",
            "require_low_loading": False,
            "exclude_top_genes": False,
            "exclude_readout_genes": False,
        })
    else:
        attempts.extend([
            {
                "label": "strict_low_loading_no_top_gene_no_readout_gene",
                "require_low_loading": True,
                "exclude_top_genes": bool(globals().get("PC_LRKO_RANDOM_EXCLUDE_TOP_LR_GENES", True)),
                "exclude_readout_genes": bool(globals().get("PC_LRKO_RANDOM_EXCLUDE_CANCERSEA_READOUT_GENES", True)),
            },
            {
                "label": "relaxed_no_low_loading_no_top_gene_no_readout_gene",
                "require_low_loading": False,
                "exclude_top_genes": bool(globals().get("PC_LRKO_RANDOM_EXCLUDE_TOP_LR_GENES", True)),
                "exclude_readout_genes": bool(globals().get("PC_LRKO_RANDOM_EXCLUDE_CANCERSEA_READOUT_GENES", True)),
            },
            {
                "label": "relaxed_no_low_loading_no_top_gene",
                "require_low_loading": False,
                "exclude_top_genes": bool(globals().get("PC_LRKO_RANDOM_EXCLUDE_TOP_LR_GENES", True)),
                "exclude_readout_genes": False,
            },
            {
                "label": "legacy_non_top_usable",
                "require_low_loading": False,
                "exclude_top_genes": False,
                "exclude_readout_genes": False,
            },
        ])

    attempt_logs = []
    for attempt in attempts:
        pool, threshold = _pc_lrko_build_random_candidate_pool(
            require_low_loading=attempt["require_low_loading"],
            exclude_top_genes=attempt["exclude_top_genes"],
            exclude_readout_genes=attempt["exclude_readout_genes"],
        )
        attempt_logs.append({
            **attempt,
            "n_candidate_pairs": int(len(pool)),
            "loading_threshold": threshold,
        })

        if len(pool) >= n_need:
            pool = list(pool)
            rng.shuffle(pool)
            return pool[:n_need], attempt["label"], attempt_logs

    raise ValueError(
        "No usable random LR baseline pairs could be sampled even after fallback attempts. "
        f"Attempt logs: {attempt_logs}"
    )


pc_lrko_random_pair_names, pc_lrko_random_selection_status, pc_lrko_random_attempt_logs = _pc_lrko_sample_random_pairs_strict()
pc_lrko_random_gene_info = _pc_lrko_lr_genes_to_indices(pc_lrko_random_pair_names, pc_lrko_gene_names)

pc_lrko_random_gene_idx_set = set(map(int, pc_lrko_random_gene_info["ligand_idx"])) | set(map(int, pc_lrko_random_gene_info["receptor_idx"]))
pc_lrko_random_overlap_top_gene_idx = sorted(pc_lrko_random_gene_idx_set & pc_lrko_top_gene_idx_set)
pc_lrko_random_overlap_readout_gene_idx = sorted(pc_lrko_random_gene_idx_set & pc_lrko_readout_gene_idx_set_for_random)

if len(pc_lrko_random_overlap_top_gene_idx) > 0:
    print("[Warning] Random LR genes overlap top-MI4 genes. Overlap gene indices:", pc_lrko_random_overlap_top_gene_idx)
if len(pc_lrko_random_overlap_readout_gene_idx) > 0:
    print("[Warning] Random LR genes overlap CancerSEA readout genes. Overlap gene indices:", pc_lrko_random_overlap_readout_gene_idx)

# Save selected LR pairs.
pc_lrko_top_lr_df = pd.DataFrame({
    "rank": np.arange(1, len(pc_lrko_top_pair_names) + 1),
    "MI_input": PC_LRKO_MI,
    "MI_name_used": pc_lrko_mi_name_used,
    "orientation_used": pc_lrko_orientation,
    "LR_pair": pc_lrko_top_pair_names,
    "raw_LR_loading": [float(pc_lrko_lr_vec.loc[x]) for x in pc_lrko_top_pair_names],
    "selection": "TopMI4",
})
# Max-normalized loadings are exported after pair selection, not used as a cutoff.
pc_lrko_top_lr_df["normalized_LR_loading"] = (
    pc_lrko_top_lr_df["raw_LR_loading"] / float(np.nanmax(pc_lrko_lr_vec.values))
)

pc_lrko_random_lr_df = pd.DataFrame({
    "rank": np.arange(1, len(pc_lrko_random_pair_names) + 1),
    "MI_input": PC_LRKO_MI,
    "MI_name_used": pc_lrko_mi_name_used,
    "orientation_used": pc_lrko_orientation,
    "LR_pair": pc_lrko_random_pair_names,
    "raw_LR_loading": [float(pc_lrko_lr_vec.loc[x]) for x in pc_lrko_random_pair_names],
    "selection": "Random",
    "random_seed": int(PC_LRKO_RANDOM_SEED),
    "random_selection_status": pc_lrko_random_selection_status,
    "random_baseline_mode": str(globals().get("PC_LRKO_RANDOM_BASELINE_MODE", "strict_low_MI4_no_overlap")),
    "exclude_top_LR_genes": bool(globals().get("PC_LRKO_RANDOM_EXCLUDE_TOP_LR_GENES", True)),
    "exclude_CancerSEA_readout_genes": bool(globals().get("PC_LRKO_RANDOM_EXCLUDE_CANCERSEA_READOUT_GENES", True)),
})
pc_lrko_random_lr_df["normalized_LR_loading"] = (
    pc_lrko_random_lr_df["raw_LR_loading"] / float(np.nanmax(pc_lrko_lr_vec.values))
)

top_lr_path = pc_mi4_lrko_outdir / f"{PC_LRKO_MI}_top{len(pc_lrko_top_pair_names)}_LR_pairs_for_LRKO.csv"
random_lr_path = pc_mi4_lrko_outdir / (
    f"{PC_LRKO_MI}_random_seed{PC_LRKO_RANDOM_SEED}_{pc_lrko_random_selection_status}_"
    f"n{len(pc_lrko_random_pair_names)}_LR_pairs_for_LRKO.csv"
)
gene_path = pc_mi4_lrko_outdir / f"{PC_LRKO_MI}_LRKO_ligand_receptor_genes.csv"

pc_lrko_top_lr_df.to_csv(top_lr_path, index=False)
pc_lrko_random_lr_df.to_csv(random_lr_path, index=False)


# Save random-baseline candidate-pool/fallback diagnostics.
pc_lrko_random_attempt_log_df = pd.DataFrame(pc_lrko_random_attempt_logs)
random_attempt_log_path = pc_mi4_lrko_outdir / (
    f"{PC_LRKO_MI}_random_seed{PC_LRKO_RANDOM_SEED}_candidate_pool_diagnostics.csv"
)
pc_lrko_random_attempt_log_df.to_csv(random_attempt_log_path, index=False)

pd.DataFrame([
    {
        "selection": "TopMI4",
        "side": "ligand",
        "n_genes_found": len(pc_lrko_top_gene_info["ligand_found"]),
        "genes_found": ";".join(pc_lrko_top_gene_info["ligand_found"]),
        "genes_missing": ";".join(pc_lrko_top_gene_info["ligand_missing"]),
    },
    {
        "selection": "TopMI4",
        "side": "receptor",
        "n_genes_found": len(pc_lrko_top_gene_info["receptor_found"]),
        "genes_found": ";".join(pc_lrko_top_gene_info["receptor_found"]),
        "genes_missing": ";".join(pc_lrko_top_gene_info["receptor_missing"]),
    },
    {
        "selection": "Random",
        "side": "ligand",
        "n_genes_found": len(pc_lrko_random_gene_info["ligand_found"]),
        "genes_found": ";".join(pc_lrko_random_gene_info["ligand_found"]),
        "genes_missing": ";".join(pc_lrko_random_gene_info["ligand_missing"]),
    },
    {
        "selection": "Random",
        "side": "receptor",
        "n_genes_found": len(pc_lrko_random_gene_info["receptor_found"]),
        "genes_found": ";".join(pc_lrko_random_gene_info["receptor_found"]),
        "genes_missing": ";".join(pc_lrko_random_gene_info["receptor_missing"]),
    },
]).to_csv(gene_path, index=False)

print("LR loading source:", pc_lrko_loading_source)
print("MI row/column used:", pc_lrko_mi_name_used)
print("Top LR pairs saved to:", top_lr_path)
display(pc_lrko_top_lr_df)
print("Random LR baseline pairs saved to:", random_lr_path)
print("Random baseline selection status:", pc_lrko_random_selection_status)
print("Random candidate-pool diagnostics saved to:", random_attempt_log_path)
display(pc_lrko_random_attempt_log_df)
display(pc_lrko_random_lr_df)

print("Top ligand genes found:", pc_lrko_top_gene_info["ligand_found"])
print("Top receptor genes found:", pc_lrko_top_gene_info["receptor_found"])
print("Random ligand genes found:", pc_lrko_random_gene_info["ligand_found"])
print("Random receptor genes found:", pc_lrko_random_gene_info["receptor_found"])


In [ ]:
# 5. Load the fixed model and define expression perturbations

from SpiderNet.model import SpiderNet_model


def _pc_lrko_extract_state_dict_from_checkpoint(checkpoint_obj):
    if isinstance(checkpoint_obj, dict) and "model_state_dict" in checkpoint_obj:
        return checkpoint_obj["model_state_dict"]
    if isinstance(checkpoint_obj, dict) and "state_dict" in checkpoint_obj:
        return checkpoint_obj["state_dict"]
    if isinstance(checkpoint_obj, dict):
        return checkpoint_obj
    raise ValueError(f"Unsupported checkpoint format: {type(checkpoint_obj)}")


def _pc_lrko_infer_spidernet_dims_from_state_dict(state_dict):
    required_keys = [
        "Loading_intrinsic_ori",
        "loading_receiver_ori",
        "loading_sender_ori",
        "loading_LR_ori",
        "enc_factor_envir_pre_receiver.0.weight",
    ]
    missing = [key for key in required_keys if key not in state_dict]
    if missing:
        raise KeyError(f"Checkpoint is missing required keys: {missing}")

    dim_intri, num_gene_from_intrinsic = state_dict["Loading_intrinsic_ori"].shape
    dim_envir, num_gene_from_receiver = state_dict["loading_receiver_ori"].shape
    dim_envir_lr, num_lr = state_dict["loading_LR_ori"].shape

    if dim_envir != dim_envir_lr:
        raise ValueError(f"Inconsistent dim_envir in checkpoint: receiver={dim_envir}, LR={dim_envir_lr}")

    if num_gene_from_intrinsic != num_gene_from_receiver:
        raise ValueError(
            "Inconsistent gene dimension in checkpoint: "
            f"intrinsic={num_gene_from_intrinsic}, receiver={num_gene_from_receiver}"
        )

    hidden_channels = int(state_dict["enc_factor_envir_pre_receiver.0.weight"].shape[0] // 2)

    return {
        "num_gene": int(num_gene_from_receiver),
        "num_LR": int(num_lr),
        "dim_intri": int(dim_intri),
        "dim_envir": int(dim_envir),
        "hidden_channels": hidden_channels,
    }


def _pc_lrko_extract_reconstruction_tensor(model_output):
    if torch.is_tensor(model_output):
        return model_output

    if isinstance(model_output, (tuple, list)) and len(model_output) > 0:
        first_item = model_output[0]
        if torch.is_tensor(first_item):
            return first_item

    if isinstance(model_output, dict):
        for key in [
            "exprecon",
            "expression_reconstruction",
            "x_recon",
            "reconstruction",
            "expr_recon",
        ]:
            if key in model_output and torch.is_tensor(model_output[key]):
                return model_output[key]

    raise ValueError("Could not identify the reconstructed expression tensor from model output.")


def _pc_lrko_ensure_model_loaded():
    global model

    device_name = globals().get("device", "cuda" if torch.cuda.is_available() else "cpu")

    if "model" in globals() and model is not None:
        model = model.to(device_name)
        model.eval()
        print("Using existing in-memory model.")
        return model

    if "reference_model_path" not in globals() or reference_model_path is None:
        raise KeyError("reference_model_path is not available. Run the model-loading cells above first.")

    checkpoint_obj = torch.load(reference_model_path, map_location="cpu")
    state_dict_obj = _pc_lrko_extract_state_dict_from_checkpoint(checkpoint_obj)
    dims_obj = _pc_lrko_infer_spidernet_dims_from_state_dict(state_dict_obj)

    if dims_obj["dim_envir"] != int(DIM_ENVIR):
        raise ValueError(
            f"DIM_ENVIR={DIM_ENVIR} does not match checkpoint dim_envir={dims_obj['dim_envir']}."
        )

    if dims_obj["num_gene"] != int(processed.genenames_train.shape[0]):
        raise ValueError(
            "Checkpoint gene dimension does not match processed.genenames_train: "
            f"checkpoint={dims_obj['num_gene']}, processed={processed.genenames_train.shape[0]}."
        )

    processed_dim_intri = int(processed.spidernet_data[0]["cell_class_onehot"].shape[1])
    if dims_obj["dim_intri"] != processed_dim_intri:
        raise ValueError(
            "Checkpoint cell-class/intrinsic dimension does not match processed data: "
            f"checkpoint={dims_obj['dim_intri']}, processed={processed_dim_intri}."
        )

    model = SpiderNet_model(
        num_gene=dims_obj["num_gene"],
        num_LR=dims_obj["num_LR"],
        hidden_channels=dims_obj["hidden_channels"],
        Factor_mode="cell_class",
        dim_intri=dims_obj["dim_intri"],
        dim_envir=dims_obj["dim_envir"],
    ).to(device_name)

    load_result = model.load_state_dict(state_dict_obj, strict=True)
    model.eval()

    print(f"Loaded pretrained model from: {reference_model_path}")
    print(load_result)

    del checkpoint_obj, state_dict_obj, dims_obj, load_result
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return model


def _pc_lrko_to_device_copy(data_obj, device_name):
    if hasattr(data_obj, "clone"):
        data_copy = data_obj.clone()
    else:
        data_copy = copy.deepcopy(data_obj)

    if hasattr(data_copy, "to"):
        data_copy = data_copy.to(device_name)

    return data_copy


def _pc_lrko_scale_graph_lr_genes(
    graph,
    sender_idx,
    receiver_idx,
    ligand_idx,
    receptor_idx,
    ligand_scale=1.0,
    receptor_scale=1.0,
):
    """Scale selected ligand genes in sender cells and receptor genes in receiver cells."""
    if hasattr(graph, "x"):
        x = graph.x
    elif isinstance(graph, dict) and "x" in graph:
        x = graph["x"]
    else:
        raise KeyError("Graph does not contain an x expression tensor.")

    x_device = x.device

    sender_idx = np.asarray(sender_idx, dtype=np.int64)
    receiver_idx = np.asarray(receiver_idx, dtype=np.int64)
    ligand_idx = np.asarray(ligand_idx, dtype=np.int64)
    receptor_idx = np.asarray(receptor_idx, dtype=np.int64)

    # Modify gene expression across selected cells; retain all graph edges.
    with torch.no_grad():
        if sender_idx.size > 0 and ligand_idx.size > 0:
            rows = torch.as_tensor(sender_idx, dtype=torch.long, device=x_device)
            cols = torch.as_tensor(ligand_idx, dtype=torch.long, device=x_device)
            x[rows[:, None], cols[None, :]] = x[rows[:, None], cols[None, :]] * float(ligand_scale)

        if receiver_idx.size > 0 and receptor_idx.size > 0:
            rows = torch.as_tensor(receiver_idx, dtype=torch.long, device=x_device)
            cols = torch.as_tensor(receptor_idx, dtype=torch.long, device=x_device)
            x[rows[:, None], cols[None, :]] = x[rows[:, None], cols[None, :]] * float(receptor_scale)

    return graph


def _pc_lrko_reconstruct_selected_gene_matrix(model_obj, graph, cell_idx, gene_idx):
    """Return reconstructed expression for selected cells and selected genes only."""
    cell_idx = np.asarray(cell_idx, dtype=np.int64)
    gene_idx = np.asarray(gene_idx, dtype=np.int64)

    if cell_idx.size == 0 or gene_idx.size == 0:
        return np.zeros((cell_idx.size, gene_idx.size), dtype=np.float32)

    with torch.no_grad():
        model_output = model_obj(graph)
        # Read total reconstructed expression, including all MI contributions.
        recon = _pc_lrko_extract_reconstruction_tensor(model_output)

        rows = torch.as_tensor(cell_idx, dtype=torch.long, device=recon.device)
        cols = torch.as_tensor(gene_idx, dtype=torch.long, device=recon.device)
        sub = recon[rows[:, None], cols[None, :]].detach().cpu().numpy().astype(np.float32, copy=False)

    del model_output, recon
    return sub


def _pc_lrko_run_reconstruction_with_lrko(
    model_obj,
    graph_original,
    sender_idx,
    receiver_idx,
    ligand_idx,
    receptor_idx,
    ligand_scale,
    receptor_scale,
    score_cell_idx,
    score_gene_idx,
):
    """Clone graph, apply LR perturbation, run SpiderNet reconstruction, and return selected reconstructed expression."""
    graph_perturb = graph_original.clone() if hasattr(graph_original, "clone") else copy.deepcopy(graph_original)
    graph_perturb = _pc_lrko_scale_graph_lr_genes(
        graph_perturb,
        sender_idx=sender_idx,
        receiver_idx=receiver_idx,
        ligand_idx=ligand_idx,
        receptor_idx=receptor_idx,
        ligand_scale=ligand_scale,
        receptor_scale=receptor_scale,
    )

    out = _pc_lrko_reconstruct_selected_gene_matrix(
        model_obj,
        graph_perturb,
        cell_idx=score_cell_idx,
        gene_idx=score_gene_idx,
    )

    del graph_perturb
    return out


pc_lrko_model = _pc_lrko_ensure_model_loaded()
pc_lrko_device = globals().get("device", "cuda" if torch.cuda.is_available() else "cpu")

print("LR-KO model ready on device:", pc_lrko_device)


In [ ]:
# 6. First pass: select tumor receivers and build unperturbed score references

# Memory-map edge-level MI activities in the exact batch/edge concatenation order
# of processed.spidernet_data. Edge counts alone cannot verify this order.
pc_lrko_factor_path = Path(run_dirs["run_dir"]) / "Factor_envir_use.npy"
if not pc_lrko_factor_path.exists():
    raise FileNotFoundError(f"Cannot find Factor_envir_use.npy: {pc_lrko_factor_path}")

pc_lrko_factor_mmap = np.load(pc_lrko_factor_path, mmap_mode="r")

pc_lrko_edge_counts = np.asarray(
    [_pc_lrko_edge_count_from_data(processed.spidernet_data[i]) for i in range(len(processed.spidernet_data))],
    dtype=np.int64,
)
pc_lrko_edge_offsets = np.concatenate(([0], np.cumsum(pc_lrko_edge_counts)))

if pc_lrko_factor_mmap.shape[0] != int(pc_lrko_edge_offsets[-1]):
    if pc_lrko_factor_mmap.ndim == 2 and pc_lrko_factor_mmap.shape[1] == int(pc_lrko_edge_offsets[-1]):
        raise ValueError(
            "Factor_envir_use appears to be MI x edge. "
            "This section expects edge x MI."
        )
    raise ValueError(
        f"Factor_envir_use shape {pc_lrko_factor_mmap.shape} is not aligned with "
        f"total edge count {pc_lrko_edge_offsets[-1]}."
    )

if PC_LRKO_MI_INDEX >= pc_lrko_factor_mmap.shape[1]:
    raise ValueError(
        f"{PC_LRKO_MI} requires column index {PC_LRKO_MI_INDEX}, "
        f"but Factor_envir_use has only {pc_lrko_factor_mmap.shape[1]} columns."
    )

# CancerSEA genes.
pc_lrko_cancersea_gene_sets = _pc_lrko_load_cancersea_gene_sets(
    programs=PC_LRKO_CANCERSEA_PROGRAMS,
    cancersea_path=PC_LRKO_CANCERSEA_PATH,
    var_names=processed.adata_list[0].var_names,
)

pc_lrko_all_gene_idx = np.unique(
    np.concatenate([v["gene_idx"] for v in pc_lrko_cancersea_gene_sets.values()])
).astype(np.int64)

pc_lrko_gene_idx_to_pos = {int(gidx): pos for pos, gidx in enumerate(pc_lrko_all_gene_idx)}

pc_lrko_program_gene_pos = {
    program: np.asarray(
        [pc_lrko_gene_idx_to_pos[int(i)] for i in info["gene_idx"]],
        dtype=np.int64,
    )
    for program, info in pc_lrko_cancersea_gene_sets.items()
}

print("CancerSEA genes used for LR-KO readout:")
for program, info in pc_lrko_cancersea_gene_sets.items():
    print(f"  {program}: {len(info['genes_used'])} genes")

# Scale expression on both ligand and receptor sides; LR-pair-count matching
# does not imply that the numbers of unique perturbed genes match.
PC_LRKO_PERTURBATIONS = OrderedDict({
    "KO-TopMI4": {
        "ligand_idx": pc_lrko_top_gene_info["ligand_idx"],
        "receptor_idx": pc_lrko_top_gene_info["receptor_idx"],
        "ligand_scale": 0.0,
        "receptor_scale": 0.0,
        "description": "Top MI4 ligand genes in Fibroblast senders and receptor genes in tumor receivers set to zero.",
    },
    "50%-TopMI4": {
        "ligand_idx": pc_lrko_top_gene_info["ligand_idx"],
        "receptor_idx": pc_lrko_top_gene_info["receptor_idx"],
        "ligand_scale": 0.5,
        "receptor_scale": 0.5,
        "description": "Top MI4 ligand/receptor genes scaled to 50%.",
    },
    "KO-Random": {
        "ligand_idx": pc_lrko_random_gene_info["ligand_idx"],
        "receptor_idx": pc_lrko_random_gene_info["receptor_idx"],
        "ligand_scale": 0.0,
        "receptor_scale": 0.0,
        "description": "Matched random ligand/receptor genes set to zero.",
    },
})

perturb_modes_path = pc_mi4_lrko_outdir / f"{PC_LRKO_MI}_LRKO_perturbation_modes.csv"
pd.DataFrame([
    {
        "Perturbation": name,
        "n_ligand_genes": len(cfg["ligand_idx"]),
        "n_receptor_genes": len(cfg["receptor_idx"]),
        "ligand_scale": cfg["ligand_scale"],
        "receptor_scale": cfg["receptor_scale"],
        "description": cfg["description"],
    }
    for name, cfg in PC_LRKO_PERTURBATIONS.items()
]).to_csv(perturb_modes_path, index=False)

# First pass: store selected receiver metadata and z-score reference statistics.
pc_lrko_selected_cell_frames = []
pc_lrko_slice_summary_rows = []
pc_lrko_selected_edge_rows = [] if PC_LRKO_SAVE_SELECTED_EDGE_TABLE else None

pc_lrko_gene_sum_global = np.zeros(len(pc_lrko_all_gene_idx), dtype=np.float64)
pc_lrko_gene_sumsq_global = np.zeros(len(pc_lrko_all_gene_idx), dtype=np.float64)
pc_lrko_gene_n_global = 0

pc_lrko_gene_sum_by_ct = {}
pc_lrko_gene_sumsq_by_ct = {}
pc_lrko_gene_n_by_ct = {}

pc_lrko_model.eval()

for sample_index, (adata, data_obj) in enumerate(zip(processed.adata_list, processed.spidernet_data)):
    if sample_index % 10 == 0:
        print(f"[LR-KO first pass] sample {sample_index + 1}/{len(processed.adata_list)}")

    edge_index = _pc_lrko_edge_index_to_numpy(_pc_lrko_get_field(data_obj, "edge_index"))
    celltypes = _pc_lrko_get_celltypes(adata, data_obj)

    if edge_index.shape[0] != int(pc_lrko_edge_counts[sample_index]):
        raise ValueError(
            f"Edge count mismatch for sample {sample_index}: "
            f"{edge_index.shape[0]} vs expected {pc_lrko_edge_counts[sample_index]}"
        )

    start = int(pc_lrko_edge_offsets[sample_index])
    end = int(pc_lrko_edge_offsets[sample_index + 1])
    mi4_edge = np.asarray(pc_lrko_factor_mmap[start:end, PC_LRKO_MI_INDEX], dtype=np.float32)

    src = edge_index[:, 0]
    dst = edge_index[:, 1]

    sender_ct = celltypes[src].astype(str)
    receiver_ct = celltypes[dst].astype(str)

    sender_is_fibroblast = sender_ct == PC_LRKO_SENDER_CELLTYPE
    receiver_is_tumor = np.char.find(receiver_ct.astype(str), PC_LRKO_RECEIVER_TUMOR_SUFFIX) >= 0
    mi4_positive = mi4_edge > float(PC_LRKO_MI_STRENGTH_THRESHOLD)

    selected_edge_mask = sender_is_fibroblast & receiver_is_tumor & mi4_positive
    selected_edge_idx = np.where(selected_edge_mask)[0].astype(np.int64)

    if selected_edge_idx.size == 0:
        pc_lrko_slice_summary_rows.append({
            "sample_index": int(sample_index),
            "sample_id": _pc_lrko_get_sample_id(adata, sample_index),
            "n_selected_edges": 0,
            "n_sender_fibroblast_cells": 0,
            "n_receiver_tumor_cells": 0,
            "mean_MI4_strength_selected_edges": np.nan,
            "status": "skip_no_Fibroblast_to_tumor_MI4_edges",
        })
        continue

    selected_edges = edge_index[selected_edge_idx, :]
    sender_idx = np.unique(selected_edges[:, 0]).astype(np.int64)
    receiver_idx = np.unique(selected_edges[:, 1]).astype(np.int64)

    receiver_celltypes = celltypes[receiver_idx].astype(str)
    receiver_cancer_types = np.asarray([
        _pc_lrko_extract_cancer_type_from_tumor_celltype(x)
        for x in receiver_celltypes
    ], dtype=object)

    valid_ct = pd.notna(receiver_cancer_types)
    receiver_idx = receiver_idx[valid_ct]
    receiver_celltypes = receiver_celltypes[valid_ct]
    receiver_cancer_types = receiver_cancer_types[valid_ct].astype(str)

    if receiver_idx.size == 0:
        pc_lrko_slice_summary_rows.append({
            "sample_index": int(sample_index),
            "sample_id": _pc_lrko_get_sample_id(adata, sample_index),
            "n_selected_edges": int(selected_edge_idx.size),
            "n_sender_fibroblast_cells": int(sender_idx.size),
            "n_receiver_tumor_cells": 0,
            "mean_MI4_strength_selected_edges": float(np.nanmean(mi4_edge[selected_edge_idx])),
            "status": "skip_no_valid_receiver_cancer_type",
        })
        continue

    receiver_barcodes = _pc_lrko_get_barcodes(adata, receiver_idx)
    sample_id = _pc_lrko_get_sample_id(adata, sample_index)

    selected_cell_df = pd.DataFrame({
        "sample_index": int(sample_index),
        "sample_id": sample_id,
        "cell_key": [
            _pc_lrko_make_cell_key(sample_index, bc)
            for bc in receiver_barcodes
        ],
        "barcode": receiver_barcodes,
        "cell_index": receiver_idx,
        "cell_role": "receiver_tumor",
        "celltype_final": receiver_celltypes,
        "CancerType": receiver_cancer_types,
        "n_incoming_Fibroblast_to_tumor_MI4_edges": np.bincount(
            dst[selected_edge_mask],
            minlength=adata.n_obs,
        )[receiver_idx].astype(np.int32),
        "incoming_Fibroblast_to_tumor_MI4_sum": np.bincount(
            dst[selected_edge_mask],
            weights=mi4_edge[selected_edge_mask],
            minlength=adata.n_obs,
        )[receiver_idx].astype(np.float32),
    })
    pc_lrko_selected_cell_frames.append(selected_cell_df)

    if PC_LRKO_SAVE_SELECTED_EDGE_TABLE:
        edge_keep_for_table = pd.DataFrame({
            "sample_index": int(sample_index),
            "sample_id": sample_id,
            "edge_index_in_sample": selected_edge_idx,
            "sender_index": selected_edges[:, 0].astype(int),
            "receiver_index": selected_edges[:, 1].astype(int),
            "sender_celltype": celltypes[selected_edges[:, 0]].astype(str),
            "receiver_celltype": celltypes[selected_edges[:, 1]].astype(str),
            "MI4_strength": mi4_edge[selected_edge_idx].astype(float),
        })
        pc_lrko_selected_edge_rows.append(edge_keep_for_table)

    # Original reconstructed expression for selected tumor receivers.
    graph_device = _pc_lrko_to_device_copy(data_obj, pc_lrko_device)
    X0 = _pc_lrko_reconstruct_selected_gene_matrix(
        pc_lrko_model,
        graph_device,
        cell_idx=receiver_idx,
        gene_idx=pc_lrko_all_gene_idx,
    )

    X0_64 = X0.astype(np.float64, copy=False)

    pc_lrko_gene_sum_global += np.nansum(X0_64, axis=0)
    pc_lrko_gene_sumsq_global += np.nansum(X0_64 ** 2, axis=0)
    pc_lrko_gene_n_global += int(X0_64.shape[0])

    for cancer_type in np.unique(receiver_cancer_types):
        rows = np.where(receiver_cancer_types == cancer_type)[0]
        if rows.size == 0:
            continue

        X_ct = X0_64[rows, :]

        if cancer_type not in pc_lrko_gene_sum_by_ct:
            pc_lrko_gene_sum_by_ct[cancer_type] = np.zeros(X_ct.shape[1], dtype=np.float64)
            pc_lrko_gene_sumsq_by_ct[cancer_type] = np.zeros(X_ct.shape[1], dtype=np.float64)
            pc_lrko_gene_n_by_ct[cancer_type] = 0

        pc_lrko_gene_sum_by_ct[cancer_type] += np.nansum(X_ct, axis=0)
        pc_lrko_gene_sumsq_by_ct[cancer_type] += np.nansum(X_ct ** 2, axis=0)
        pc_lrko_gene_n_by_ct[cancer_type] += int(X_ct.shape[0])

    pc_lrko_slice_summary_rows.append({
        "sample_index": int(sample_index),
        "sample_id": sample_id,
        "n_selected_edges": int(selected_edge_idx.size),
        "n_sender_fibroblast_cells": int(sender_idx.size),
        "n_receiver_tumor_cells": int(receiver_idx.size),
        "mean_MI4_strength_selected_edges": float(np.nanmean(mi4_edge[selected_edge_idx])),
        "status": "used",
    })

    del graph_device, X0, X0_64
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

if len(pc_lrko_selected_cell_frames) == 0:
    raise ValueError("No Fibroblast->tumor MI4 receiver tumor cells were selected.")

pc_lrko_selected_cells_df = pd.concat(pc_lrko_selected_cell_frames, axis=0, ignore_index=True)
pc_lrko_slice_summary_df = pd.DataFrame(pc_lrko_slice_summary_rows)

pc_lrko_gene_ref_global = _pc_lrko_build_gene_reference(
    pc_lrko_gene_sum_global,
    pc_lrko_gene_sumsq_global,
    pc_lrko_gene_n_global,
)

pc_lrko_gene_ref_by_ct = {}
for cancer_type in sorted(pc_lrko_gene_sum_by_ct.keys()):
    pc_lrko_gene_ref_by_ct[cancer_type] = _pc_lrko_build_gene_reference(
        pc_lrko_gene_sum_by_ct[cancer_type],
        pc_lrko_gene_sumsq_by_ct[cancer_type],
        pc_lrko_gene_n_by_ct[cancer_type],
    )

# Save receiver metadata, marker coverage, and reference cell counts.
# Gene-wise reference means/stds remain in memory for the second pass.
selected_cells_path = pc_mi4_lrko_outdir / f"{PC_LRKO_MI}_LRKO_selected_receiver_tumor_cells.csv"
slice_summary_path = pc_mi4_lrko_outdir / f"{PC_LRKO_MI}_LRKO_slice_summary.csv"
gene_counts_path = pc_mi4_lrko_outdir / f"{PC_LRKO_MI}_LRKO_CancerSEA_genes_used.csv"
ref_summary_path = pc_mi4_lrko_outdir / f"{PC_LRKO_MI}_LRKO_CancerSEA_zscore_reference_summary_{PC_LRKO_MODULE_SCORE_METHOD}.csv"

pc_lrko_selected_cells_df.to_csv(selected_cells_path, index=False)
pc_lrko_slice_summary_df.to_csv(slice_summary_path, index=False)

if PC_LRKO_SAVE_SELECTED_EDGE_TABLE and len(pc_lrko_selected_edge_rows) > 0:
    selected_edges_path = pc_mi4_lrko_outdir / f"{PC_LRKO_MI}_LRKO_selected_Fibroblast_to_tumor_edges.csv"
    pd.concat(pc_lrko_selected_edge_rows, axis=0, ignore_index=True).to_csv(selected_edges_path, index=False)
    print("Saved selected-edge table:", selected_edges_path)

pd.DataFrame([
    {
        "Program": program,
        "n_genes_used": len(info["genes_used"]),
        "genes_used": ";".join(info["genes_used"]),
        "n_genes_missing": len(info["genes_missing"]),
        "genes_missing": ";".join(info["genes_missing"]),
    }
    for program, info in pc_lrko_cancersea_gene_sets.items()
]).to_csv(gene_counts_path, index=False)

ref_rows = [{
    "reference": "global",
    "CancerType": "all",
    "n_ref": int(pc_lrko_gene_ref_global["n_ref"]),
    "module_score_method": PC_LRKO_MODULE_SCORE_METHOD,
}]
for cancer_type, ref in pc_lrko_gene_ref_by_ct.items():
    ref_rows.append({
        "reference": str(cancer_type),
        "CancerType": str(cancer_type),
        "n_ref": int(ref["n_ref"]),
        "module_score_method": PC_LRKO_MODULE_SCORE_METHOD,
    })
pd.DataFrame(ref_rows).to_csv(ref_summary_path, index=False)

print("Selected receiver tumor cells by CancerType:")
display(
    pc_lrko_selected_cells_df
    .groupby("CancerType", as_index=False)
    .agg(n_cells=("cell_key", "nunique"))
    .sort_values("CancerType")
)

print("Saved selected cells:", selected_cells_path)
print("Saved slice summary:", slice_summary_path)
print("Saved CancerSEA gene table:", gene_counts_path)
print("Saved z-score reference summary:", ref_summary_path)

display(pc_lrko_slice_summary_df.head())


In [ ]:
# 7. Second pass: reconstruct perturbations and calculate program-score changes

def _pc_lrko_compute_program_scores(X, cancer_types):
    """
    Compute CancerSEA module scores from selected-gene expression matrix.
    Uses the global or within-cancer-type reference built in the first pass.
    """
    X = np.asarray(X, dtype=np.float32)
    cancer_types = np.asarray(cancer_types).astype(str)

    out = {
        program: np.full(X.shape[0], np.nan, dtype=np.float32)
        for program in PC_LRKO_CANCERSEA_PROGRAMS
    }

    # Reuse the unperturbed reference for every perturbation.
    if PC_LRKO_MODULE_SCORE_METHOD == "zscore_mean_global":
        ref = pc_lrko_gene_ref_global
        X_z = (X - ref["mean"][None, :]) / ref["std"][None, :]

        for program in PC_LRKO_CANCERSEA_PROGRAMS:
            cols = pc_lrko_program_gene_pos[program]
            out[program] = np.nanmean(X_z[:, cols], axis=1).astype(np.float32)

    elif PC_LRKO_MODULE_SCORE_METHOD == "zscore_mean_within_cancertype":
        for cancer_type in np.unique(cancer_types):
            if cancer_type not in pc_lrko_gene_ref_by_ct:
                continue

            rows = np.where(cancer_types == cancer_type)[0]
            if rows.size == 0:
                continue

            ref = pc_lrko_gene_ref_by_ct[cancer_type]
            X_z = (X[rows, :] - ref["mean"][None, :]) / ref["std"][None, :]

            for program in PC_LRKO_CANCERSEA_PROGRAMS:
                cols = pc_lrko_program_gene_pos[program]
                out[program][rows] = np.nanmean(X_z[:, cols], axis=1).astype(np.float32)

    else:
        raise ValueError(f"Unsupported PC_LRKO_MODULE_SCORE_METHOD={PC_LRKO_MODULE_SCORE_METHOD}.")

    return out


pc_lrko_go_change_frames = []
pc_lrko_second_pass_summary_rows = []

selected_by_sample = {
    int(sample_index): df.reset_index(drop=True)
    for sample_index, df in pc_lrko_selected_cells_df.groupby("sample_index", sort=True)
}

for sample_index, meta_sub in selected_by_sample.items():
    if sample_index % 10 == 0:
        print(f"[LR-KO second pass] sample {sample_index + 1}/{len(processed.adata_list)}")

    adata = processed.adata_list[sample_index]
    data_obj = processed.spidernet_data[sample_index]

    edge_index = _pc_lrko_edge_index_to_numpy(_pc_lrko_get_field(data_obj, "edge_index"))
    celltypes = _pc_lrko_get_celltypes(adata, data_obj)

    start = int(pc_lrko_edge_offsets[sample_index])
    end = int(pc_lrko_edge_offsets[sample_index + 1])
    mi4_edge = np.asarray(pc_lrko_factor_mmap[start:end, PC_LRKO_MI_INDEX], dtype=np.float32)

    src = edge_index[:, 0]
    dst = edge_index[:, 1]

    sender_ct = celltypes[src].astype(str)
    receiver_ct = celltypes[dst].astype(str)

    selected_edge_mask = (
        (sender_ct == PC_LRKO_SENDER_CELLTYPE)
        & (np.char.find(receiver_ct.astype(str), PC_LRKO_RECEIVER_TUMOR_SUFFIX) >= 0)
        & (mi4_edge > float(PC_LRKO_MI_STRENGTH_THRESHOLD))
    )

    selected_edges = edge_index[selected_edge_mask, :]
    if selected_edges.shape[0] == 0:
        continue

    sender_idx = np.unique(selected_edges[:, 0]).astype(np.int64)
    receiver_idx = meta_sub["cell_index"].to_numpy(dtype=np.int64)
    cancer_types = meta_sub["CancerType"].astype(str).to_numpy()

    graph_device = _pc_lrko_to_device_copy(data_obj, pc_lrko_device)

    # Original reconstruction for the same selected receiver tumor cells.
    X_original = _pc_lrko_reconstruct_selected_gene_matrix(
        pc_lrko_model,
        graph_device,
        cell_idx=receiver_idx,
        gene_idx=pc_lrko_all_gene_idx,
    )
    original_scores = _pc_lrko_compute_program_scores(X_original, cancer_types)

    base_meta = meta_sub[
        [
            "cell_key",
            "sample_index",
            "sample_id",
            "barcode",
            "cell_index",
            "cell_role",
            "celltype_final",
            "CancerType",
            "n_incoming_Fibroblast_to_tumor_MI4_edges",
            "incoming_Fibroblast_to_tumor_MI4_sum",
        ]
    ].copy()
    base_meta["MI"] = PC_LRKO_MI

    for perturb_name, perturb_cfg in PC_LRKO_PERTURBATIONS.items():
        t0 = time.time()
        X_perturb = _pc_lrko_run_reconstruction_with_lrko(
            pc_lrko_model,
            graph_device,
            sender_idx=sender_idx,
            receiver_idx=receiver_idx,
            ligand_idx=perturb_cfg["ligand_idx"],
            receptor_idx=perturb_cfg["receptor_idx"],
            ligand_scale=perturb_cfg["ligand_scale"],
            receptor_scale=perturb_cfg["receptor_scale"],
            score_cell_idx=receiver_idx,
            score_gene_idx=pc_lrko_all_gene_idx,
        )
        perturb_scores = _pc_lrko_compute_program_scores(X_perturb, cancer_types)

        for program in PC_LRKO_CANCERSEA_PROGRAMS:
            cur_df = base_meta.copy()
            cur_df["GO_program"] = program
            cur_df["Perturbation"] = perturb_name
            cur_df["n_CancerSEA_genes_in_data"] = int(len(pc_lrko_program_gene_pos[program]))
            cur_df["original_module_score"] = original_scores[program].astype(np.float32)
            cur_df["perturbed_module_score"] = perturb_scores[program].astype(np.float32)
            cur_df["delta_module_score"] = (
                perturb_scores[program] - original_scores[program]
            ).astype(np.float32)
            cur_df["module_score_method"] = PC_LRKO_MODULE_SCORE_METHOD
            pc_lrko_go_change_frames.append(cur_df)

        pc_lrko_second_pass_summary_rows.append({
            "sample_index": int(sample_index),
            "sample_id": str(meta_sub["sample_id"].iloc[0]),
            "Perturbation": perturb_name,
            "n_sender_fibroblast_cells": int(sender_idx.size),
            "n_receiver_tumor_cells": int(receiver_idx.size),
            "elapsed_min": float((time.time() - t0) / 60.0),
        })

        del X_perturb, perturb_scores
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    del graph_device, X_original, original_scores, base_meta
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

if len(pc_lrko_go_change_frames) == 0:
    raise ValueError("No LR-KO CancerSEA module-score changes were computed.")

pc_mi4_lrko_cancersea_delta_long = pd.concat(
    pc_lrko_go_change_frames,
    axis=0,
    ignore_index=True,
)
pc_mi4_lrko_second_pass_summary_df = pd.DataFrame(pc_lrko_second_pass_summary_rows)

delta_long_path = pc_mi4_lrko_outdir / (
    f"{PC_LRKO_MI}_Fibroblast_to_tumor_LRKO_CancerSEA_delta_long_"
    f"{PC_LRKO_MODULE_SCORE_METHOD}.csv"
)
run_time_path = pc_mi4_lrko_outdir / f"{PC_LRKO_MI}_Fibroblast_to_tumor_LRKO_runtime_by_sample.csv"

pc_mi4_lrko_cancersea_delta_long.to_csv(delta_long_path, index=False)
pc_mi4_lrko_second_pass_summary_df.to_csv(run_time_path, index=False)

print("Saved LR-KO CancerSEA delta long table:", delta_long_path)
print("Saved LR-KO runtime summary:", run_time_path)
display(pc_mi4_lrko_cancersea_delta_long.head())
display(pc_mi4_lrko_second_pass_summary_df.head())

# Release per-sample chunk list after concatenation to save memory.
del pc_lrko_go_change_frames
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
# 8. Export descriptive summaries and unpaired control comparisons

def _pc_lrko_wilcoxon_against_zero(values):
    values = pd.to_numeric(pd.Series(values), errors="coerce").dropna().to_numpy(dtype=float)

    if values.size < 2 or np.nanstd(values) <= 0:
        return np.nan

    try:
        return float(wilcoxon(values, alternative="two-sided").pvalue)
    except Exception:
        return np.nan


summary_rows = []

group_cols = ["CancerType", "GO_program", "Perturbation"]

for keys, df_cur in pc_mi4_lrko_cancersea_delta_long.groupby(group_cols, sort=False):
    cancer_type, program, perturb = keys
    delta = pd.to_numeric(df_cur["delta_module_score"], errors="coerce").dropna().to_numpy(dtype=float)

    summary_rows.append({
        "CancerType": cancer_type,
        "GO_program": program,
        "Perturbation": perturb,
        "n_cells": int(delta.size),
        "mean_delta_module_score": float(np.nanmean(delta)) if delta.size > 0 else np.nan,
        "median_delta_module_score": float(np.nanmedian(delta)) if delta.size > 0 else np.nan,
        "std_delta_module_score": float(np.nanstd(delta, ddof=1)) if delta.size > 1 else np.nan,
        "wilcoxon_p_delta_vs_zero": _pc_lrko_wilcoxon_against_zero(delta),
        "fraction_delta_below_zero": float(np.mean(delta < 0)) if delta.size > 0 else np.nan,
        "fraction_delta_above_zero": float(np.mean(delta > 0)) if delta.size > 0 else np.nan,
    })

# P-values in these exports are unadjusted.
pc_mi4_lrko_cancersea_summary = pd.DataFrame(summary_rows)

# This summary uses an unpaired two-sided Mann-Whitney U comparison.
# The final cell separately tests paired receiver-cell differences.
baseline_compare_rows = []

for (cancer_type, program), df_cur in pc_mi4_lrko_cancersea_delta_long.groupby(["CancerType", "GO_program"], sort=False):
    top_vals = pd.to_numeric(
        df_cur.loc[df_cur["Perturbation"] == "KO-TopMI4", "delta_module_score"],
        errors="coerce",
    ).dropna().to_numpy(dtype=float)
    random_vals = pd.to_numeric(
        df_cur.loc[df_cur["Perturbation"] == "KO-Random", "delta_module_score"],
        errors="coerce",
    ).dropna().to_numpy(dtype=float)

    if len(top_vals) > 0 and len(random_vals) > 0:
        try:
            p_top_vs_random = float(mannwhitneyu(top_vals, random_vals, alternative="two-sided").pvalue)
        except Exception:
            p_top_vs_random = np.nan
    else:
        p_top_vs_random = np.nan

    baseline_compare_rows.append({
        "CancerType": cancer_type,
        "GO_program": program,
        "n_KO_TopMI4": int(len(top_vals)),
        "n_KO_Random": int(len(random_vals)),
        "mean_delta_KO_TopMI4": float(np.nanmean(top_vals)) if len(top_vals) > 0 else np.nan,
        "mean_delta_KO_Random": float(np.nanmean(random_vals)) if len(random_vals) > 0 else np.nan,
        "median_delta_KO_TopMI4": float(np.nanmedian(top_vals)) if len(top_vals) > 0 else np.nan,
        "median_delta_KO_Random": float(np.nanmedian(random_vals)) if len(random_vals) > 0 else np.nan,
        "mean_delta_Top_minus_Random": (
            float(np.nanmean(top_vals) - np.nanmean(random_vals))
            if len(top_vals) > 0 and len(random_vals) > 0 else np.nan
        ),
        "median_delta_Top_minus_Random": (
            float(np.nanmedian(top_vals) - np.nanmedian(random_vals))
            if len(top_vals) > 0 and len(random_vals) > 0 else np.nan
        ),
        "mannwhitney_p_KO_TopMI4_vs_KO_Random": p_top_vs_random,
    })

pc_mi4_lrko_top_vs_random_summary = pd.DataFrame(baseline_compare_rows)

summary_path = pc_mi4_lrko_outdir / (
    f"{PC_LRKO_MI}_Fibroblast_to_tumor_LRKO_CancerSEA_delta_summary_by_cancertype_"
    f"{PC_LRKO_MODULE_SCORE_METHOD}.csv"
)
top_vs_random_path = pc_mi4_lrko_outdir / (
    f"{PC_LRKO_MI}_Fibroblast_to_tumor_LRKO_CancerSEA_KOTop_vs_KORandom_by_cancertype_"
    f"{PC_LRKO_MODULE_SCORE_METHOD}.csv"
)

pc_mi4_lrko_cancersea_summary.to_csv(summary_path, index=False)
pc_mi4_lrko_top_vs_random_summary.to_csv(top_vs_random_path, index=False)

print("Saved LR-KO summary:", summary_path)
print("Saved KO-TopMI4 vs KO-Random summary:", top_vs_random_path)

display(pc_mi4_lrko_cancersea_summary.head(12))
display(pc_mi4_lrko_top_vs_random_summary.head(12))


In [ ]:
# 9. Plot receiver-cell program changes by cancer type

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

plt.close("all")
plt.style.use("default")
mpl.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "Arial",
    "font.size": 7,
    "axes.labelsize": 7,
    "axes.titlesize": 8,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "axes.linewidth": 0.7,
    "xtick.major.width": 0.7,
    "ytick.major.width": 0.7,
    "xtick.major.size": 2.5,
    "ytick.major.size": 2.5,
    "figure.dpi": 150,
    "savefig.dpi": 300,
})
sns.set_theme(style="white")

PC_LRKO_PLOT_PERTURB_ORDER = [
    "KO-TopMI4",
    "50%-TopMI4",
    "KO-Random",
]

PC_LRKO_PLOT_PALETTE = {
    "KO-TopMI4": "#8ca9ff",
    "50%-TopMI4": "#8dd4ff",
    "KO-Random": "#cccccc",
}

PC_LRKO_PROGRAM_ORDER = ["Invasion", "Angiogenesis", "Hypoxia"]

# An existing CANCERTYPE_ORDER overrides the alphabetical order of a fresh kernel.
if "CANCERTYPE_ORDER" in globals():
    pc_lrko_cancer_type_order = [
        ct for ct in CANCERTYPE_ORDER
        if ct in set(pc_mi4_lrko_cancersea_delta_long["CancerType"].astype(str))
    ]
    pc_lrko_extra_ct = sorted(
        set(pc_mi4_lrko_cancersea_delta_long["CancerType"].astype(str))
        - set(pc_lrko_cancer_type_order)
    )
    pc_lrko_cancer_type_order = pc_lrko_cancer_type_order + pc_lrko_extra_ct
else:
    pc_lrko_cancer_type_order = sorted(
        pc_mi4_lrko_cancersea_delta_long["CancerType"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )


def _pc_lrko_save_fig_both(fig, basepath_no_ext):
    basepath_no_ext = str(basepath_no_ext)
    fig.savefig(basepath_no_ext + ".pdf", bbox_inches="tight", facecolor="white")
    fig.savefig(basepath_no_ext + ".png", bbox_inches="tight", facecolor="white", dpi=300)


def _pc_lrko_panel_ylim(panel_df, q_low=1, q_high=99, pad=1.15):
    """
    Determine adaptive y-axis range for one panel only.

    Values are pooled across the three perturbation groups within
    the current CancerType × CancerSEA-program panel.
    """
    vals = pd.to_numeric(
        panel_df["delta_module_score"],
        errors="coerce",
    ).replace([np.inf, -np.inf], np.nan).dropna().to_numpy(dtype=float)

    if vals.size == 0:
        return (-1.0, 1.0)

    # Percentiles set display limits; the underlying plotting data are retained.
    lo, hi = np.nanpercentile(vals, [q_low, q_high])

    # Keep zero visible and use a small margin.
    lo = min(float(lo), 0.0)
    hi = max(float(hi), 0.0)

    span = hi - lo
    if span <= 1e-6:
        span = max(abs(lo), abs(hi), 1e-3)

    return (lo - pad * 0.08 * span, hi + pad * 0.08 * span)


def _pc_lrko_draw_vertical_violin_box(ax, values, x_pos, color):
    vals = pd.to_numeric(pd.Series(values), errors="coerce").dropna().to_numpy(dtype=float)
    vals = vals[np.isfinite(vals)]

    if vals.size == 0:
        return

    if vals.size >= 2 and np.nanstd(vals) > 0:
        violin_parts = ax.violinplot(
            vals,
            positions=[x_pos],
            vert=True,
            widths=0.72,
            showmeans=False,
            showmedians=False,
            showextrema=False,
        )
        for body in violin_parts["bodies"]:
            body.set_facecolor(color)
            body.set_edgecolor("#939393")
            body.set_linewidth(0.7)
            body.set_alpha(1.0)
    else:
        ax.scatter(
            np.full(vals.shape[0], x_pos),
            vals,
            s=5,
            color=color,
            edgecolors="none",
            zorder=3,
        )

    bp = ax.boxplot(
        vals,
        positions=[x_pos],
        vert=True,
        widths=0.18,
        patch_artist=True,
        showfliers=False,
        showcaps=False,
        boxprops={
            "facecolor": "black",
            "edgecolor": "black",
            "linewidth": 0.7,
        },
        whiskerprops={
            "color": "black",
            "linewidth": 0.7,
        },
        medianprops={
            "color": "white",
            "linewidth": 0.9,
        },
    )

    for key in ["boxes", "whiskers", "medians"]:
        for artist in bp[key]:
            artist.set_zorder(5)


plot_df = pc_mi4_lrko_cancersea_delta_long[
    pc_mi4_lrko_cancersea_delta_long["GO_program"].isin(PC_LRKO_PROGRAM_ORDER)
    & pc_mi4_lrko_cancersea_delta_long["Perturbation"].isin(PC_LRKO_PLOT_PERTURB_ORDER)
    & pc_mi4_lrko_cancersea_delta_long["CancerType"].astype(str).isin(pc_lrko_cancer_type_order)
].copy()

plot_df["GO_program"] = pd.Categorical(
    plot_df["GO_program"].astype(str),
    categories=PC_LRKO_PROGRAM_ORDER,
    ordered=True,
)
plot_df["Perturbation"] = pd.Categorical(
    plot_df["Perturbation"].astype(str),
    categories=PC_LRKO_PLOT_PERTURB_ORDER,
    ordered=True,
)
plot_df["CancerType"] = pd.Categorical(
    plot_df["CancerType"].astype(str),
    categories=pc_lrko_cancer_type_order,
    ordered=True,
)

plot_df["delta_module_score"] = pd.to_numeric(
    plot_df["delta_module_score"],
    errors="coerce",
)
plot_df = plot_df.replace([np.inf, -np.inf], np.nan)
plot_df = plot_df.dropna(
    subset=["GO_program", "Perturbation", "CancerType", "delta_module_score"]
)

if plot_df.shape[0] == 0:
    raise ValueError("No data available for LR-KO CancerSEA plotting.")

n_rows = len(PC_LRKO_PROGRAM_ORDER)
n_cols = len(pc_lrko_cancer_type_order)

fig_width = max(10.5, 1.35 * n_cols + 1.8)
fig_height = max(5.0, 1.35 * n_rows + 1.2)

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(fig_width, fig_height),
    sharex=True,
    sharey=False,
    facecolor="white",
)

if n_rows == 1:
    axes = np.asarray([axes])
if n_cols == 1:
    axes = axes.reshape(n_rows, 1)

for r, program in enumerate(PC_LRKO_PROGRAM_ORDER):
    for c, cancer_type in enumerate(pc_lrko_cancer_type_order):
        ax = axes[r, c]
        ax.set_facecolor("white")

        panel_df = plot_df[
            (plot_df["GO_program"] == program)
            & (plot_df["CancerType"] == cancer_type)
        ].copy()

        for x_pos, perturb in enumerate(PC_LRKO_PLOT_PERTURB_ORDER):
            vals = panel_df.loc[
                panel_df["Perturbation"] == perturb,
                "delta_module_score",
            ]

            _pc_lrko_draw_vertical_violin_box(
                ax=ax,
                values=vals,
                x_pos=x_pos,
                color=PC_LRKO_PLOT_PALETTE[perturb],
            )

        ax.axhline(0, color="#4D4D4D", linewidth=0.65, linestyle="--", zorder=0)

        # Each panel has its own adaptive y-axis range.
        ax.set_ylim(_pc_lrko_panel_ylim(panel_df))

        ax.set_xticks(np.arange(len(PC_LRKO_PLOT_PERTURB_ORDER)))

        if r == n_rows - 1:
            ax.set_xticklabels(
                PC_LRKO_PLOT_PERTURB_ORDER,
                rotation=45,
                ha="right",
                rotation_mode="anchor",
            )
            ax.set_xlabel("")
        else:
            ax.set_xticklabels([])
            ax.set_xlabel("")

        if c == 0:
            ax.set_ylabel(
                f"{program}\nΔ module score",
                fontsize=8,
                fontweight="bold",
            )
        else:
            ax.set_ylabel("")

        if r == 0:
            ax.set_title(str(cancer_type), fontsize=8, pad=4)

        ax.tick_params(axis="both", length=2.5, width=0.7, pad=1.5)

        sns.despine(ax=ax, top=True, right=True)

legend_handles = [
    mpl.patches.Patch(
        facecolor=PC_LRKO_PLOT_PALETTE[p],
        edgecolor="#939393",
        linewidth=0.7,
        label=p,
    )
    for p in PC_LRKO_PLOT_PERTURB_ORDER
]

fig.legend(
    handles=legend_handles,
    title="Perturbation",
    frameon=False,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.02),
    ncol=len(PC_LRKO_PLOT_PERTURB_ORDER),
    fontsize=7,
    title_fontsize=7,
)

fig.suptitle(
    f"{PC_LRKO_MI} Fibroblast→tumor top LR-pair KO effect on tumor-cell CancerSEA programs",
    y=1.08,
    fontsize=9,
)

plt.tight_layout(w_pad=0.6, h_pad=0.8)

plot_table_path = pc_mi4_lrko_outdir / (
    f"{PC_LRKO_MI}_Fibroblast_to_tumor_LRKO_CancerSEA_plotting_table_"
    f"{PC_LRKO_MODULE_SCORE_METHOD}.csv"
)
plot_df.to_csv(plot_table_path, index=False)

fig_prefix = pc_mi4_lrko_outdir / (
    f"{PC_LRKO_MI}_Fibroblast_to_tumor_LRKO_CancerSEA_"
    f"3x{n_cols}_vertical_violin_boxplot_delta_module_score_"
    f"{PC_LRKO_MODULE_SCORE_METHOD}_panel_scaled"
)
_pc_lrko_save_fig_both(fig, fig_prefix)

plt.show()
plt.close(fig)

print("Saved LR-KO plotting table:", plot_table_path)
print("Saved LR-KO CancerSEA vertical violin+boxplot to:", str(fig_prefix) + ".pdf/.png")

In [ ]:
# 10. Export per-panel top-LR versus random-LR tests
from scipy.stats import mannwhitneyu, wilcoxon

# Use paired two-sided Wilcoxon signed-rank tests when receiver-cell keys match.
# Otherwise use the recorded unpaired Mann-Whitney U fallback. P-values are unadjusted.
pc_lrko_panel_test_rows = []

if "cell_key" in plot_df.columns:
    pc_lrko_pair_key_cols = ["cell_key"]
elif {"sample_index", "cell_index"}.issubset(plot_df.columns):
    pc_lrko_pair_key_cols = ["sample_index", "cell_index"]
else:
    pc_lrko_pair_key_cols = []

for program in PC_LRKO_PROGRAM_ORDER:
    for cancer_type in pc_lrko_cancer_type_order:
        panel_df = plot_df.loc[
            (plot_df["GO_program"].astype(str) == str(program))
            & (plot_df["CancerType"].astype(str) == str(cancer_type))
            & plot_df["Perturbation"].astype(str).isin(["KO-TopMI4", "KO-Random"]),
        ].copy()
        panel_df["delta_module_score"] = pd.to_numeric(
            panel_df["delta_module_score"], errors="coerce"
        )
        panel_df = panel_df.replace([np.inf, -np.inf], np.nan).dropna(
            subset=["delta_module_score"]
        )

        top_df = panel_df.loc[
            panel_df["Perturbation"].astype(str) == "KO-TopMI4"
        ].copy()
        random_df = panel_df.loc[
            panel_df["Perturbation"].astype(str) == "KO-Random"
        ].copy()

        top_all = top_df["delta_module_score"].to_numpy(dtype=float)
        random_all = random_df["delta_module_score"].to_numpy(dtype=float)
        paired = pd.DataFrame()
        pairing_is_valid = False
        pairing_status = "pairing key unavailable"

        if pc_lrko_pair_key_cols:
            top_pair = top_df[pc_lrko_pair_key_cols + ["delta_module_score"]].copy()
            random_pair = random_df[pc_lrko_pair_key_cols + ["delta_module_score"]].copy()
            top_pair = top_pair.rename(columns={"delta_module_score": "KO_TopMI4"})
            random_pair = random_pair.rename(columns={"delta_module_score": "KO_Random"})

            top_has_duplicate_keys = top_pair.duplicated(pc_lrko_pair_key_cols).any()
            random_has_duplicate_keys = random_pair.duplicated(pc_lrko_pair_key_cols).any()

            if not top_has_duplicate_keys and not random_has_duplicate_keys:
                paired = top_pair.merge(
                    random_pair, on=pc_lrko_pair_key_cols, how="inner", validate="one_to_one"
                ).dropna(subset=["KO_TopMI4", "KO_Random"])
                pairing_is_valid = len(paired) >= 2
                pairing_status = (
                    "one-to-one matched receiver cells"
                    if pairing_is_valid else "fewer than two matched receiver cells"
                )
            else:
                pairing_status = "duplicate rows for pairing key"

        statistic = np.nan
        p_value = np.nan
        status = "ok"

        if pairing_is_valid:
            top_test = paired["KO_TopMI4"].to_numpy(dtype=float)
            random_test = paired["KO_Random"].to_numpy(dtype=float)
            paired_difference = top_test - random_test
            test_name = "Wilcoxon signed-rank"
            paired_test = True

            if np.allclose(paired_difference, 0):
                statistic, p_value = 0.0, 1.0
                status = "all paired differences are zero"
            else:
                try:
                    test_result = wilcoxon(
                        top_test, random_test, alternative="two-sided",
                        zero_method="wilcox", method="auto",
                    )
                except TypeError:
                    test_result = wilcoxon(
                        top_test, random_test, alternative="two-sided",
                        zero_method="wilcox",
                    )
                statistic = float(test_result.statistic)
                p_value = float(test_result.pvalue)

            n_paired = int(len(paired))
            median_top = float(np.median(top_test))
            median_random = float(np.median(random_test))
            median_difference = float(np.median(paired_difference))
        else:
            test_name = "Wilcoxon rank-sum (Mann-Whitney U)"
            paired_test = False
            n_paired = int(len(paired))
            median_top = float(np.median(top_all)) if len(top_all) else np.nan
            median_random = float(np.median(random_all)) if len(random_all) else np.nan
            median_difference = (
                median_top - median_random
                if np.isfinite(median_top) and np.isfinite(median_random) else np.nan
            )

            if len(top_all) > 0 and len(random_all) > 0:
                try:
                    test_result = mannwhitneyu(
                        top_all, random_all, alternative="two-sided", method="auto"
                    )
                except TypeError:
                    test_result = mannwhitneyu(
                        top_all, random_all, alternative="two-sided"
                    )
                statistic = float(test_result.statistic)
                p_value = float(test_result.pvalue)
                status = f"rank-sum fallback: {pairing_status}"
            else:
                status = f"test unavailable: missing one or both groups; {pairing_status}"

        pc_lrko_panel_test_rows.append({
            "CancerType": str(cancer_type),
            "GO_program": str(program),
            "comparison": "KO-TopMI4 vs KO-Random",
            "alternative": "two-sided",
            "test": test_name,
            "paired": paired_test,
            "pairing_key": " + ".join(pc_lrko_pair_key_cols) if pc_lrko_pair_key_cols else "",
            "n_KO_TopMI4": int(len(top_all)),
            "n_KO_Random": int(len(random_all)),
            "n_paired": n_paired,
            "statistic": statistic,
            "p_value": p_value,
            "median_delta_KO_TopMI4": median_top,
            "median_delta_KO_Random": median_random,
            "median_delta_Top_minus_Random": median_difference,
            "status": status,
        })

pc_mi4_lrko_panel_top_vs_random_tests = pd.DataFrame(pc_lrko_panel_test_rows)

panel_test_path = pc_mi4_lrko_outdir / (
    f"{PC_LRKO_MI}_Fibroblast_to_tumor_LRKO_CancerSEA_"
    f"KOTop_vs_KORandom_per_panel_two_sided_tests_{PC_LRKO_MODULE_SCORE_METHOD}.csv"
)
pc_mi4_lrko_panel_top_vs_random_tests.to_csv(panel_test_path, index=False)

print("Per-panel KO-TopMI4 versus KO-Random two-sided tests:")
display(pc_mi4_lrko_panel_top_vs_random_tests)
print("Saved per-panel test results:", panel_test_path)
